# Modellphase
## 1. EBM Modell (inherently interpretable model)

### Schritt 1: Das Problem für EBM übersetzen (Klassifikation statt Ranking)

Da EBM von Natur aus kein "Ranking-Modell" ist (es kann Startlisten nicht direkt vergleichen), müssen wir das Problem für die Phase A in eine binäre Klassifikation (Ja/Nein-Frage) umwandeln.

- Die Frage an das Modell lautet: „Landet dieser spezifische Fahrer auf dieser spezifischen Etappe in den Top 3/10/20?“

- Das Target (y): Aus der Spalte rank machen wir eine neue Zielvariable (z. B. is_top_3). Wenn ein Fahrer Platz 1, 2 oder 3 belegt hat, bekommt er eine 1 (Ja), alle anderen Fahrer (Platz 4 bis 160) bekommen eine 0 (Nein).

### Schritt 2: Die zeitlich korrekte Datentrennung (Chronological Split)

Im Radsport darf man die Daten niemals zufällig mischen (Random Split). Wenn das Modell mit Daten aus der Tour de France 2025 trainiert wird, darf es nicht die Tour de France 2010 vorhersagen – das wäre in der Wissenschaft ein schwerer methodischer Fehler (Data Leakage durch Zeit).

- Trainings-Daten: Wir füttern das Modell mit allen historischen Etappen von 2005 bis einschließlich 2023. Hier lernt das EBM die Muster.

- Test-/Validierungs-Daten: Wir halten die Jahre 2024 und 2025 komplett unter Verschluss. Sie dienen als unsere "Zukunft", an der wir später prüfen, wie gut das Modell wirklich ist.

### Schritt 3: Die Metadaten maskieren

Wenn wir das EBM trainieren, darf es nur mit echten, mathematischen Features lernen. Spalten wie der Klartextname des Fahrers, das Team oder die Etappen-URL würden das Modell verwirren oder zum Absturz bringen.
D.h. wir maskieren alle Spalten mit dem meta_prefix.


### Schritt 4: Das Training der "Glass-Box" (EBM)

Jetzt wird das EBM-Modell auf den Trainingsdaten (2005–2023) gestartet.

- Wie es intern lernt: EBM schaut sich jedes Feature (wie gradient_final_km, rider_bmi, weather_temp_mean) komplett isoliert an und baut für jedes einzelne Feature eine mathematische Kurve (einen sogenannten Shape Plot). 
- Es lernt zum Beispiel: „Wenn die Steigung gegen 0% geht, steigt die Wahrscheinlichkeit für Sprinter. 
- Wenn sie über 7% steigt, stürzt ihre Wahrscheinlichkeit ab und die der Kletterer schießt nach oben.“

- Da es keine unkontrollierten Interaktionen zwischen den Features erlaubt, bleibt dieses gelernte Wissen zu 100% transparent und nachvollziehbar.

### Schritt 5: Die Brücke zum Ranking bauen (Post-Processing)

Wenn das Modell trainiert ist, jagen wir unsere Testdaten (die Jahre 2024 & 2025) durch das EBM.

- Das EBM schaut sich die Startliste von beispielsweise Etappe 14 der Tour de France 2024 an.

- Für jeden einzelnen Fahrer auf dieser Startliste berechnet das EBM eine exakte Gewinn-Wahrscheinlichkeit zwischen 0% und 100%.

- Der Ranking-Trick: Wir nehmen diese berechneten Wahrscheinlichkeiten und sortieren die Startliste dieser Etappe in Pandas einfach abwärts (höchste Wahrscheinlichkeit zuerst).

- Das Ergebnis: Der Fahrer mit der höchsten EBM-Wahrscheinlichkeit wird unser prognostizierter Platz 1, der zweite Platz 2, usw. Damit haben wir aus einem Klassifikationsmodell ein fertiges Etappen-Ranking gebaut!

### Schritt 6: Die primäre Erklärung herbeiführen (Das Dozenten-Highlight)

Zum Schluss nutzen wir die integrierte Visualisierung der EBM-Bibliothek (interpret). Wir lassen uns die gelernten Kurven (Shape Plots) der wichtigsten Features ausgeben – allen voran für euer brandneues Feature gradient_final_km.


In [37]:
import os
import pandas as pd
import numpy as np
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from interpret import show
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

In [3]:
pickle_path = '../../data/processed/24_cleaned_master_data.pkl'

df = pd.read_pickle(pickle_path)

### Schritt 1 Schritt 1: Das Problem für EBM übersetzen (Klassifikation statt Ranking)

In [4]:
df['rank'].dtype
df['rank'].astype(int)

0           1
1           2
2           3
3           4
4           5
         ... 
196043    149
196044    150
196045    151
196046    152
196047    153
Name: rank, Length: 196048, dtype: int64

In [ ]:
wetter_spalten = [
    'wind_stability_index',
    'weather_temp_mean',
    'weather_temp_trend',
    'weather_rain_prob_mean',
    'weather_precipitation_mean',
    'weather_humidity_mean'
]

for col in wetter_spalten:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [14]:
# Wir erstellen das Top-5/10 Target (1 = Top5/10/20, 0 = Plätze 6+/10+/20+)
df['target_top_5'] = np.where(df['rank'] <= 5, 1, 0)

df['target_top_10'] = np.where(df['rank'] <= 10, 1, 0)

df['target_top_20'] = np.where(df['rank'] <= 20, 1, 0)

In [ ]:
print(df['target_top_5'].value_counts().to_string())
print(df['target_top_10'].value_counts().to_string())
print(df['target_top_20'].value_counts().to_string())


# Verteilung der 1 und 0 Werte

target_top_5
0    190197
1      5851
target_top_10
0    184368
1     11680
target_top_20
0    172740
1     23308


### Schritt 2: Die zeitlich korrekte Datentrennung (Chronological Split) + Schritt 3: Die Metadaten maskieren

In [33]:
chosen_target = 'target_top_10'
# Für ersten Lauf top 10 nutzen

# Automatische Feature-Selektion:
# Wir filtern alle Spalten heraus, die mit 'meta_' beginnen oder eines der Targets sind.
features = [col for col in df.columns if not col.startswith('meta_')
            and col not in ['rank', 'rank_numeric', 'target_top_5', 'target_top_10', 'target_top_20'] and 'won_how' not in col]

print(f"Selektierte Lern-Features {len(features)} Stück für das EBM-Modell:")
print()

# --- TRAININGS-DATEN (Historie: 2005 bis einschließlich 2023) ---
df_train_historical = df[df['meta_year'] <= 2023]
X_train = df_train_historical[features]
y_train = df_train_historical[chosen_target]

# --- TEST-DATEN (Zukunft: Saisons 2024 und 2025) ---
df_test_future = df[df['meta_year'] >= 2024]
X_test = df_test_future[features]
y_test = df_test_future[chosen_target]

print("ERGEBNIS DES CHRONOLOGISCHEN SPLITS:")
print(f"[TRAIN] X_train (Features 2005-2023): {X_train.shape[0]} Zeilen x {X_train.shape[1]} Spalten")
print(f"[TRAIN] y_train (Labels   2005-2023): {y_train.shape[0]} Zeilen")
print(f"[TEST] X_test  (Features 2024-2025): {X_test.shape[0]} Zeilen x {X_test.shape[1]} Spalten")
print(f"[TEST] y_test  (Labels   2024-2025): {y_test.shape[0]} Zeilen")


Selektierte Lern-Features 25 Stück für das EBM-Modell:

ERGEBNIS DES CHRONOLOGISCHEN SPLITS:
[TRAIN] X_train (Features 2005-2023): 178246 Zeilen x 25 Spalten
[TRAIN] y_train (Labels   2005-2023): 178246 Zeilen
[TEST] X_test  (Features 2024-2025): 17802 Zeilen x 25 Spalten
[TEST] y_test  (Labels   2024-2025): 17802 Zeilen


### Schritt 4: Das Training der "Glass-Box" (EBM)

In [ ]:
# 1. Wir berechnen die Gewichte händisch mit Scikit-Learn, um das Klassenungleichgewicht auszugleichen
# Da nur ca. 6-7% in den Top 10 landen, bekommen die '1er' ein höheres Gewicht als die '0er'
sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

# 2. Initialisieren des EBM
ebm = ExplainableBoostingClassifier(
    random_state=42,
    n_jobs=-1  # Nutzt alle Prozessorkerne deines PCs für maximale Geschwindigkeit
)

# 3. Wir übergeben die berechneten Gewichte direkt beim mathematischen Training
ebm.fit(X_train, y_train, sample_weight=sample_weights_train)

,feature_names,None
,feature_types,None
,max_bins,1024
,max_interaction_bins,64
,interactions,'3x'
,exclude,None
,validation_size,0.15
,outer_bags,14
,inner_bags,0
,learning_rate,0.015
,greedy_ratio,10.0


In [46]:
show(ebm.explain_global())

<!-- http://127.0.0.1:7001/2162630105408/ -->

### Schritt 5: Die Brücke zum Ranking bauen (Post-Processing)

In [47]:
# Wir berechnen die Wahrscheinlichkeiten (Wahrscheinlichkeit für Klasse 1, also Top 10)
# predict_proba gibt [Wahrscheinlichkeit für 0, Wahrscheinlichkeit für 1] aus, daher [:, 1]
y_pred_proba = ebm.predict_proba(X_test)[:, 1]

# Wir machen auch eine harte Ja/Nein-Vorhersage (0 oder 1) basierend auf dem Standard-Threshold
y_pred_binary = ebm.predict(X_test)

# Wir hängen die Vorhersagen an das originale Test-DataFrame mit den Metadaten an!
# Dadurch verknüpfen wir die mathematische Vorhersage wieder mit Fahrernamen und Etappen.
df_results = df_test_future.copy()
df_results['ebm_prob'] = y_pred_proba
df_results['ebm_pred_binary'] = y_pred_binary

In [48]:
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score auf den Testdaten (2024-2025): {auc_score:.4f}")

ROC-AUC Score auf den Testdaten (2024-2025): 0.8683


In [49]:
# 2. Classification Report (Precision, Recall, F1-Score)
print("Detaillierter Klassifikations-Report:")
print(classification_report(y_test, y_pred_binary))

Detaillierter Klassifikations-Report:
              precision    recall  f1-score   support

           0       0.98      0.75      0.85     16698
           1       0.18      0.82      0.30      1104

    accuracy                           0.76     17802
   macro avg       0.58      0.79      0.57     17802
weighted avg       0.93      0.76      0.82     17802



In [50]:
print("Konfusionsmatrix")
print(confusion_matrix(y_test, y_pred_binary))

Konfusionsmatrix
[[12586  4112]
 [  198   906]]


### 4 + 5 ohne die Features aus PCS (Onedayraces, GC, TT, Sprint, Climer, Hills)

In [ ]:
intransparente_pcs_features = [
    'climber', 'sprint', 'hills', 'gc', 'time_trial', 'one_day_races'
]

features_pure = [col for col in df.columns if not col.startswith('meta_')
                 and 'target' not in col
                 and 'rank' not in col
                 and 'won_how' not in col
                 and col not in intransparente_pcs_features]


X_train_pure = df_train_historical[features_pure]
X_test_pure = df_test_future[features_pure]


In [ ]:
ebm_pure = ExplainableBoostingClassifier(
    random_state=42,
    n_jobs=-1
)

ebm_pure.fit(X_train_pure, y_train, sample_weight=sample_weights_train)

,feature_names,None
,feature_types,None
,max_bins,1024
,max_interaction_bins,64
,interactions,'3x'
,exclude,None
,validation_size,0.15
,outer_bags,14
,inner_bags,0
,learning_rate,0.015
,greedy_ratio,10.0


In [55]:
show(ebm_pure.explain_global())

<!-- http://127.0.0.1:7001/2162742860096/ -->

In [56]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# 1. Wahrscheinlichkeiten für die Testdaten (Modell B) berechnen
y_pred_proba_pure = ebm_pure.predict_proba(X_test_pure)[:, 1]

# 2. Harte Ja/Nein-Vorhersagen (0 oder 1) für den Classification Report berechnen
y_pred_binary_pure = ebm_pure.predict(X_test_pure)

# 3. ROC-AUC Score berechnen und ausgeben
auc_score_pure = roc_auc_score(y_test, y_pred_proba_pure)
print(f"ROC-AUC Score (Modell B): {auc_score_pure:.4f}")
print()

# 4. Detaillierter Classification Report (Precision, Recall, F1-Score)
print("Detaillierter Klassifikations-Report für Modell B:")
print(classification_report(y_test, y_pred_binary_pure))
print()

# 5. Confusion Matrix ausgeben
print("Konfusionsmatrix für Modell B:")
print(confusion_matrix(y_test, y_pred_binary_pure))


ROC-AUC Score (Modell B): 0.8294

Detaillierter Klassifikations-Report für Modell B:
              precision    recall  f1-score   support

           0       0.98      0.72      0.83     16698
           1       0.15      0.78      0.26      1104

    accuracy                           0.72     17802
   macro avg       0.57      0.75      0.54     17802
weighted avg       0.93      0.72      0.79     17802


Konfusionsmatrix für Modell B:
[[12010  4688]
 [  245   859]]


### 4 + 5 ohne die Features aus PCS (Onedayraces, GC, TT, Sprint, Climer, Hills, rider_points_season)

In [ ]:
leakage_punkte_features = [
    'climber', 'sprint', 'hills', 'gc', 'time_trial', 'cobbles', 'one_day_races',
    'rider_rank_season',
    'rider_points_season', 'rider_points_per_season'
]

# Wir filtern die Features erneut, schließen aber JEDE Spalte aus, die "points" oder "rank" im Namen hat
features_bulletproof = [col for col in df.columns if not col.startswith('meta_')
                        and 'target' not in col
                        and 'rank' not in col
                        and 'won_how_cat' not in col
                        and 'points' not in col  # Wirft alle aktuellen Punktespalten radikal raus
                        and col not in leakage_punkte_features]

In [58]:
X_train_c = df_train_historical[features_bulletproof]
X_test_c = df_test_future[features_bulletproof]

In [ ]:
ebm_c = ExplainableBoostingClassifier(
    random_state=42,
    n_jobs=-1
)

ebm_c.fit(X_train_c, y_train, sample_weight=sample_weights_train)

In [ ]:
show(ebm_c.explain_global())

In [ ]:
y_pred_proba_c = ebm_c.predict_proba(X_test_c)[:, 1]
y_pred_binary_c = ebm_c.predict(X_test_c)